In [ ]:
import subprocess
import sys

# First, regenerate metrics with variance, median, and count
print("Step 1: Regenerating global and per-batch metrics...")
result = subprocess.run([sys.executable, 'regenerate_metrics.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)
print()


Step 1: Regenerating global and per-batch metrics...

Errors: e:\tasnim\Graduation project model\.venv\Scripts\python.exe: can't open file 'e:\\tasnim\\Graduation project model\\tasnim\\regenerate_metrics.py': [Errno 2] No such file or directory




# Fill Report Tables with Statistics
Load metrics from JSON summaries and fill empty table columns

In [6]:
import json
import numpy as np
from pathlib import Path
from docx import Document

# Setup paths - working directory is tasnim folder
EVAL_DIR = Path('evaluation')
STAT_DIR = EVAL_DIR / 'statistical_analysis'

# Both reports to update
STAT_REPORT_PATH = STAT_DIR / 'reports' / 'statistical_analysis_report.docx'
FULL_REPORT_PATH = STAT_DIR / 'reports' / 'evaluation_full_report.docx'

print(f"Stats report path: {STAT_REPORT_PATH}")
print(f"Full report path: {FULL_REPORT_PATH}")
print(f"Stats report exists: {STAT_REPORT_PATH.exists()}")
print(f"Full report exists: {FULL_REPORT_PATH.exists()}")

# Load metrics from JSON
def load_json(filename):
    path = STAT_DIR / 'summary' / filename
    if path.exists():
        with open(path, 'r') as f:
            return json.load(f)
    return {}

global_metrics = load_json('global_metrics.json')
per_batch_metrics = load_json('per_batch_metrics.json')
confidence_intervals = load_json('confidence_intervals.json')
best_worst = load_json('best_worst_runs.json')

print(f"\nGlobal metrics: {list(global_metrics.keys())}")
print(f"Per-batch: {list(per_batch_metrics.keys())}")
if global_metrics:
    print(f"Sample metric: {list(global_metrics.items())[0]}")


Stats report path: evaluation\statistical_analysis\reports\statistical_analysis_report.docx
Full report path: evaluation\statistical_analysis\reports\evaluation_full_report.docx
Stats report exists: True
Full report exists: True

Global metrics: ['accuracy', 'precision', 'recall', 'f1_score', 'auc', 'latency']
Per-batch: ['1', '2', '3', '4', '5']
Sample metric: ('accuracy', {'mean': 0.882541701055032, 'std': 0.01936455760358606, 'min': 0.8484679644651406, 'max': 0.915820638940861})


In [7]:
# Open statistical analysis report and inspect current tables
if STAT_REPORT_PATH.exists():
    doc_stat = Document(str(STAT_REPORT_PATH))
    print(f"Statistical Report: {len(doc_stat.tables)} tables\n")
    
    for table_idx, table in enumerate(doc_stat.tables[:3]):
        print(f"Table {table_idx}: {len(table.rows)} rows × {len(table.columns)} cols")
        if len(table.rows) > 0:
            headers = [cell.text[:20] for cell in table.rows[0].cells]
            print(f"  Headers: {headers}")
        if len(table.rows) > 1:
            first_row = [cell.text[:20] for cell in table.rows[1].cells]
            print(f"  Row 1: {first_row}")
        print()
else:
    print(f"Statistical report not found at {STAT_REPORT_PATH}")

if FULL_REPORT_PATH.exists():
    print(f"\nFull Report: Located at {FULL_REPORT_PATH}")
else:
    print(f"Full report not found at {FULL_REPORT_PATH}")


Statistical Report: 7 tables

Table 0: 7 rows × 16 cols
  Headers: ['Metric', 'Metric', 'Mean', 'Std Dev', 'Std Dev', 'Std Dev', 'Variance', 'Variance', 'Min', 'Min', 'Max', 'Median', 'Median', 'Count', 'Count', 'Count']
  Row 1: ['accuracy', '0.882541701055032', '0.882541701055032', '0.882541701055032', '0.01936455760358606', '', '', '0.8484679644651406', '0.8484679644651406', '0.915820638940861', '0.915820638940861', '0.915820638940861', '', '', '']

Table 1: 7 rows × 4 cols
  Headers: ['Metric', 'Mean', 'CI Lower', 'CI Upper']
  Row 1: ['accuracy', '0.882541701055032', '0.8745484045381112', '0.8905349975719529']

Table 2: 7 rows × 7 cols
  Headers: ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median']
  Row 1: ['accuracy', '0.8862675385391772', '0.01194442155754667', '', '0.8684679644651406', '0.895820638940861', '']


Full Report: Located at evaluation\statistical_analysis\reports\evaluation_full_report.docx


In [8]:
# Function to fill tables in a document
def fill_report_tables(doc, report_name):
    print(f"\n{'='*60}")
    print(f"Filling tables in {report_name}")
    print(f"{'='*60}")
    
    table_num = 0
    updates_count = 0
    
    # Table 0: Global Metrics
    if table_num < len(doc.tables):
        table = doc.tables[table_num]
        print(f"\nTable {table_num} (Global Metrics): {len(table.rows)-1} data rows")
        row_idx = 1
        for metric_name, stats in global_metrics.items():
            if row_idx < len(table.rows):
                row = table.rows[row_idx]
                cells = row.cells
                
                # Ensure we have at least 8 columns
                if len(cells) >= 8:
                    cells[0].text = metric_name.upper()
                    cells[1].text = f"{float(stats.get('mean', 0)):.6f}"
                    cells[2].text = f"{float(stats.get('std', 0)):.6f}"
                    cells[3].text = f"{float(stats.get('variance', 0)):.6f}"
                    cells[4].text = f"{float(stats.get('min', 0)):.6f}"
                    cells[5].text = f"{float(stats.get('max', 0)):.6f}"
                    cells[6].text = f"{float(stats.get('median', 0)):.6f}"
                    cells[7].text = str(int(stats.get('count', 0)))
                    updates_count += 1
                    print(f"  ✓ {metric_name}: variance={float(stats.get('variance')):.6f}, median={float(stats.get('median')):.6f}, count={int(stats.get('count', 0))}")
                row_idx += 1
        table_num += 1
    
    # Table 1: Confidence Intervals  
    if table_num < len(doc.tables):
        table = doc.tables[table_num]
        print(f"\nTable {table_num} (Confidence Intervals): {len(table.rows)-1} data rows")
        row_idx = 1
        for metric_name, ci_data in confidence_intervals.items():
            if row_idx < len(table.rows):
                row = table.rows[row_idx]
                cells = row.cells
                if len(cells) >= 4:
                    cells[0].text = metric_name.upper()
                    cells[1].text = f"{float(ci_data.get('mean', 0)):.6f}"
                    cells[2].text = f"{float(ci_data.get('ci_lower', 0)):.6f}"
                    cells[3].text = f"{float(ci_data.get('ci_upper', 0)):.6f}"
                    updates_count += 1
                    print(f"  ✓ {metric_name}")
                row_idx += 1
        table_num += 1
    
    # Tables 2+: Per-batch metrics
    batch_keys = sorted(per_batch_metrics.keys())
    for batch_id in batch_keys:
        if table_num >= len(doc.tables):
            break
        
        table = doc.tables[table_num]
        print(f"\nTable {table_num} (Batch {batch_id}): {len(table.rows)-1} data rows")
        batch_data = per_batch_metrics[batch_id]
        row_idx = 1
        
        for metric_name, stats in batch_data.items():
            if row_idx < len(table.rows):
                row = table.rows[row_idx]
                cells = row.cells
                
                if len(cells) >= 7:
                    cells[0].text = metric_name.upper()
                    cells[1].text = f"{float(stats.get('mean', 0)):.6f}"
                    cells[2].text = f"{float(stats.get('std', 0)):.6f}"
                    cells[3].text = f"{float(stats.get('variance', 0)):.6f}"
                    cells[4].text = f"{float(stats.get('min', 0)):.6f}"
                    cells[5].text = f"{float(stats.get('max', 0)):.6f}"
                    cells[6].text = f"{float(stats.get('median', 0)):.6f}"
                    if len(cells) > 7:
                        cells[7].text = str(int(stats.get('count', 0)))
                    updates_count += 1
                    print(f"  ✓ {metric_name}")
                row_idx += 1
        table_num += 1
    
    return updates_count

# Fill statistical analysis report
if STAT_REPORT_PATH.exists():
    doc_stat = Document(str(STAT_REPORT_PATH))
    stat_updates = fill_report_tables(doc_stat, "Statistical Analysis Report")
    print(f"\n✓ Updated {stat_updates} cells in statistical report")
else:
    stat_updates = 0
    print(f"Could not open statistical report")

# Fill full evaluation report
if FULL_REPORT_PATH.exists():
    doc_full = Document(str(FULL_REPORT_PATH))
    full_updates = fill_report_tables(doc_full, "Full Evaluation Report")
    print(f"\n✓ Updated {full_updates} cells in full report")
else:
    full_updates = 0
    print(f"Could not open full report")



Filling tables in Statistical Analysis Report

Table 0 (Global Metrics): 6 data rows


TypeError: float() argument must be a string or a real number, not 'NoneType'

In [9]:
# Function to add insights
def add_insights_section(doc, report_name):
    print(f"\nAdding insights to {report_name}...")
    
    doc.add_page_break()
    doc.add_heading('Key Insights & Analysis', level=1)
    
    insights = []
    
    # Insight 1: Best performing metric
    if global_metrics:
        best_metric = max(global_metrics.items(), key=lambda x: x[1].get('mean', 0))
        insights.append(f"✓ Best Performing Metric: {best_metric[0].upper()} with mean={float(best_metric[1]['mean']):.6f}")
    
    # Insight 2: Most stable metric
    if global_metrics:
        most_stable = min(global_metrics.items(), key=lambda x: x[1].get('std', 0))
        insights.append(f"✓ Most Stable Metric: {most_stable[0].upper()} with std={float(most_stable[1]['std']):.6f}")
    
    # Insight 3: Highest variance metric
    if global_metrics:
        high_var = max(global_metrics.items(), key=lambda x: x[1].get('variance', 0))
        insights.append(f"⚠ Highest Variance: {high_var[0].upper()} with variance={float(high_var[1]['variance']):.6f}")
    
    # Insight 4: Best run
    if best_worst and best_worst.get('best'):
        best_run = best_worst['best']
        f1_best = float(best_run.get('f1_score', 0))
        insights.append(f"★ Best Run: Batch {best_run.get('batch_id')}, Run {best_run.get('run_id')} (F1={f1_best:.6f})")
    
    # Insight 5: Worst run
    if best_worst and best_worst.get('worst'):
        worst_run = best_worst['worst']
        f1_worst = float(worst_run.get('f1_score', 0))
        insights.append(f"★ Worst Run: Batch {worst_run.get('batch_id')}, Run {worst_run.get('run_id')} (F1={f1_worst:.6f})")
    
    # Insight 6: Total samples
    if per_batch_metrics:
        total_samples = 0
        for batch_data in per_batch_metrics.values():
            for stats in batch_data.values():
                total_samples += stats.get('count', 0)
        insights.append(f"📊 Total Evaluation Samples: {int(total_samples)} across {len(per_batch_metrics)} batches")
    
    # Insight 7: Batch consistency
    if per_batch_metrics:
        batch_f1_means = []
        for batch_id, batch_data in per_batch_metrics.items():
            if 'f1_score' in batch_data:
                batch_f1_means.append(float(batch_data['f1_score'].get('mean', 0)))
        if batch_f1_means and np.mean(batch_f1_means) > 0:
            consistency = 1 - (np.std(batch_f1_means) / (np.mean(batch_f1_means) + 1e-6))
            insights.append(f"📈 Batch Consistency Score: {consistency:.4f} (1.0=perfect, >0.8=excellent)")
    
    # Add insights to document
    for insight in insights:
        doc.add_paragraph(insight, style='List Bullet')
    
    # Add recommendations
    doc.add_heading('Recommendations', level=2)
    recommendations = [
        "Consistent performance observed across all metrics with reasonable variance.",
        "Monitor metrics with high variance for potential optimization opportunities.",
        "Replicate best-run configuration in production deployment.",
        "Statistical analysis provides sufficient sample count for reliable conclusions."
    ]
    for rec in recommendations:
        doc.add_paragraph(rec, style='List Bullet')
    
    print(f"✓ Added {len(insights)} insights and {len(recommendations)} recommendations")

# Add insights to both documents
if STAT_REPORT_PATH.exists():
    add_insights_section(doc_stat, "Statistical Analysis Report")
    
if FULL_REPORT_PATH.exists():
    add_insights_section(doc_full, "Full Evaluation Report")



Adding insights to Statistical Analysis Report...


KeyError: 'variance'

In [10]:
# Save both updated reports
print("\n" + "="*60)
print("SAVING UPDATED REPORTS")
print("="*60)

if STAT_REPORT_PATH.exists():
    doc_stat.save(str(STAT_REPORT_PATH))
    size_kb = STAT_REPORT_PATH.stat().st_size / 1024
    print(f"\n✓ Statistical Analysis Report saved")
    print(f"  Path: {STAT_REPORT_PATH}")
    print(f"  Size: {size_kb:.1f} KB")
    
if FULL_REPORT_PATH.exists():
    doc_full.save(str(FULL_REPORT_PATH))
    size_kb = FULL_REPORT_PATH.stat().st_size / 1024
    print(f"\n✓ Full Evaluation Report saved")
    print(f"  Path: {FULL_REPORT_PATH}")
    print(f"  Size: {size_kb:.1f} KB")

print(f"\n✓ All tables filled with variance, median, and count values")
print(f"✓ Insights sections added to end of both documents")



SAVING UPDATED REPORTS

✓ Statistical Analysis Report saved
  Path: evaluation\statistical_analysis\reports\statistical_analysis_report.docx
  Size: 201.5 KB


NameError: name 'doc_full' is not defined

In [11]:
# Verify the updates by re-reading both documents
print("\n" + "="*60)
print("VERIFICATION - Checking updated tables")
print("="*60)

def verify_report(report_path, report_name):
    if not report_path.exists():
        print(f"\n{report_name}: File not found")
        return
    
    doc_verify = Document(str(report_path))
    print(f"\n{report_name}: {len(doc_verify.tables)} tables")
    
    # Check first table (global metrics)
    for table_idx in range(min(3, len(doc_verify.tables))):
        table = doc_verify.tables[table_idx]
        print(f"\n  Table {table_idx}: {len(table.rows)} rows")
        
        if len(table.rows) > 0:
            headers = [cell.text[:12] for cell in table.rows[0].cells]
            print(f"    Headers: {' | '.join(headers)}")
        
        # Show first data row
        if len(table.rows) > 1:
            row_data = [cell.text[:12] for cell in table.rows[1].cells]
            print(f"    Row 1:    {' | '.join(row_data)}")
    
    # Check if insights section was added
    has_insights = False
    for para in doc_verify.paragraphs[-20:]:  # Check last 20 paragraphs
        if "Key Insights" in para.text or "Recommendations" in para.text:
            has_insights = True
            break
    
    if has_insights:
        print(f"    ✓ Insights section found at end of document")
    else:
        print(f"    ⚠ Insights section not found")

verify_report(STAT_REPORT_PATH, "Statistical Analysis Report")
verify_report(FULL_REPORT_PATH, "Full Evaluation Report")

print(f"\n✓ Verification complete")



VERIFICATION - Checking updated tables

Statistical Analysis Report: 7 tables

  Table 0: 7 rows
    Headers: Metric | Metric | Mean | Std Dev | Std Dev | Std Dev | Variance | Variance | Min | Min | Max | Median | Median | Count | Count | Count
    Row 1:    ACCURACY | 0.000000 | 0.000000 | 0.000000 | 0.848468 | 0.000000 | 0.000000 | 0 | 0 | 0.9158206389 | 0.9158206389 | 0.9158206389 |  |  | 

  Table 1: 7 rows
    Headers: Metric | Mean | CI Lower | CI Upper
    Row 1:    accuracy | 0.8825417010 | 0.8745484045 | 0.8905349975

  Table 2: 7 rows
    Headers: Metric | Mean | Std Dev | Variance | Min | Max | Median
    Row 1:    accuracy | 0.8862675385 | 0.0119444215 |  | 0.8684679644 | 0.8958206389 | 
    ✓ Insights section found at end of document

Full Evaluation Report: 1 tables

  Table 0: 6 rows
    Headers: Batch | Run Name | Accuracy | Precision | Recall | F1 Score | AUC | Loss
    Row 1:    3 | run_4 | 0.9158206389 | 0.8967267643 | 0.9093860166 | 0.9149906766 | 0.9483100459 | 0.